- Measure accuracy for devices
- Home settings

In [1]:
from __future__ import annotations
import pandas as pd 
from typing import Dict, List, Tuple
from pathlib import Path
from collections.abc import Iterable
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import time
import re
import numpy as np
from copy import deepcopy
from datetime import datetime
from typing import Any
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [2]:
DATASET_FOLDER = "../../../dataset/processed-data"
DEVICE_MAPPING_FOLDER = DATASET_FOLDER + "/device-mapping"
INPUT_FOLDER = DATASET_FOLDER + "/input-data"
DATASET1 = "dataset1"
DATASET2 = "dataset2"

In [3]:
MAPPING_LIST_1 = f"{DEVICE_MAPPING_FOLDER}/{DATASET1}/list_of_devices_processed.csv"
MAPPING_LIST_2 = f"{DEVICE_MAPPING_FOLDER}/{DATASET2}/list_of_devices_processed.csv"
INPUT_1 = f"{INPUT_FOLDER}/{DATASET1}"
INPUT_2 = f"{INPUT_FOLDER}/{DATASET2}"

In [4]:
csv_files_1 = [f"{INPUT_1}/{f.name}" for f in sorted(Path(INPUT_1).iterdir()) if f.is_file() and f.suffix == ".csv"]
for file in csv_files_1:
    print(file)

../../../dataset/processed-data/input-data/dataset1/16-09-23.csv
../../../dataset/processed-data/input-data/dataset1/16-09-24.csv
../../../dataset/processed-data/input-data/dataset1/16-09-25.csv
../../../dataset/processed-data/input-data/dataset1/16-09-26.csv
../../../dataset/processed-data/input-data/dataset1/16-09-27.csv
../../../dataset/processed-data/input-data/dataset1/16-09-28.csv
../../../dataset/processed-data/input-data/dataset1/16-09-29.csv
../../../dataset/processed-data/input-data/dataset1/16-09-30.csv
../../../dataset/processed-data/input-data/dataset1/16-10-01.csv
../../../dataset/processed-data/input-data/dataset1/16-10-02.csv
../../../dataset/processed-data/input-data/dataset1/16-10-03.csv
../../../dataset/processed-data/input-data/dataset1/16-10-04.csv
../../../dataset/processed-data/input-data/dataset1/16-10-05.csv
../../../dataset/processed-data/input-data/dataset1/16-10-06.csv
../../../dataset/processed-data/input-data/dataset1/16-10-07.csv
../../../dataset/processe

In [5]:
device_mapping_1 = pd.read_csv(MAPPING_LIST_1)
device_dict_1 = (
    device_mapping_1
    .set_index("mac_address")
    .drop(columns=["mac_address"], errors="ignore")
    .to_dict(orient="index")
)

In [6]:
FEATURE_COLUMNS = [
    "packet_count",
    "total_bytes",
]

DEVICE_COLUMN = "eth.src"          # Replace with "eth.src" if applicable
TIME_COLUMN = "timestamp"
LABEL_COLUMN = "is_iot_device"

KPLITS = 5
RANDOM_STATE = 42
WINDOW_SECONDS = 4

In [7]:
# Outgoing volume
def extract_data_from_pcap(
    file_paths: Iterable[str]
) -> pd.DataFrame:
    columns = [
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "eth.src",
        "eth.dst",
    ]

    all_data = []

    for file_path in file_paths:
        raw_df = pd.read_csv(
            file_path,
            usecols=columns,
            dtype={
                "frame.number": "int32",
                "frame.time_epoch": "float64",
                "frame.len": "int32",
                "eth.src": "category",
                "eth.dst": "category",
            }
        )

        raw_df["capture_id"] = Path(file_path).stem
        all_data.append(raw_df)


    res = pd.concat(all_data, ignore_index=True)
    res = res.sort_values(
        ["frame.time_epoch", "eth.src"],
        ignore_index=True,
    )

    return res


In [8]:
all_data = extract_data_from_pcap(csv_files_1)

In [9]:
def label_features(
    features: pd.DataFrame,
    device_mapping: Dict[str, Dict[str, str]]
) -> pd.DataFrame:

    unknown_macs = set(features["eth.src"]) - set(device_mapping)

    if unknown_macs:
        raise KeyError(
            f"MAC addresses not found in device_mapping: {unknown_macs}"
        )

    res = features.copy()
    res["is_iot_device"] = res["eth.src"].map(
        lambda x: device_mapping[x]["is_iot_device"]
    )
    res["device_type"] = res["eth.src"].map(
        lambda x: device_mapping[x]["device_type"]
    )
    res["vendor"] = res["eth.src"].map(
        lambda x: device_mapping[x]["vendor"]
    )

    return res

In [10]:
df = all_data.copy()

- A lot of traffic that are Ipv6 multicast neighbor discovery like 33:33:ff:xx:xx:xx. We'll classify if a mac is broadcast, unicast, or multicast
- If the mac are all FF -> broadcast
- Least significant bit of the first octect = 0 -> unicast
- Least significant bit of the first octect = 1 -> multicast

In [12]:
def classify_mac(mac):
    if pd.isna(mac):
        return "invalid"

    mac = str(mac).lower().replace("-", ":")
    parts = mac.split(":")

    if len(parts) != 6:
        return "invalid"

    try:
        octets = [int(x, 16) for x in parts]
    except ValueError:
        return "invalid"

    if octets == [0xff] * 6:
        return "broadcast"

    if octets[0] & 0x01:
        return "multicast"

    return "unicast"

In [13]:
# device_info = {}

# for mac in device_dict_1:
#     source_mac = mac

#     agg_df = (
#         df[df["eth.src"] == source_mac]
#         .dropna(subset=["eth.dst"])
#         .groupby("eth.dst", observed=True)
#         .agg(
#             packet_count=("frame.len", "size"),
#             total_bytes=("frame.len", "sum")
#         )
#         .reset_index()
#         .sort_values("packet_count", ascending=False)
#     )

#     # Percentage of total outgoing packets/bytes
#     agg_df["packet_count_pct"] = (
#         agg_df["packet_count"] / agg_df["packet_count"].sum() * 100
#     )

#     agg_df["total_bytes_pct"] = (
#         agg_df["total_bytes"] / agg_df["total_bytes"].sum() * 100
#     )

#     agg_df["dst_type"] = agg_df["eth.dst"].apply(classify_mac)

#     print(
#         f"Traffic from "
#         f"{device_dict_1[source_mac]['device_name']} "
#         f"({source_mac})"
#     )

#     device_info[source_mac] = {
#         "device_name": device_dict_1[source_mac]["device_name"],
#         "traffic_summary": agg_df
#     }

#     display(agg_df.sort_values("dst_type"))
device_info = {}
for mac in device_dict_1:
    source_mac = mac

    agg_df = (
        df[df["eth.src"] == source_mac]
        .dropna(subset=["eth.dst"])
        .groupby("eth.dst", observed=True)
        .agg(
            packet_count=("frame.len", "size"),
            total_bytes=("frame.len", "sum")
        )
        .reset_index()
        .sort_values("packet_count", ascending=False)
    )

    agg_df["dst_type"] = agg_df["eth.dst"].apply(classify_mac)

    # Drop unicast destinations that are not known devices
    agg_df = agg_df[
        ~(
            (agg_df["dst_type"] == "unicast")
            & (~agg_df["eth.dst"].isin(device_dict_1.keys()))
        )
    ].copy()

    # Percentage of total outgoing packets/bytes
    agg_df["packet_count_pct"] = (
        agg_df["packet_count"]
        / agg_df["packet_count"].sum()
        * 100
    )

    agg_df["total_bytes_pct"] = (
        agg_df["total_bytes"]
        / agg_df["total_bytes"].sum()
        * 100
    )

    print(
        f"Traffic from "
        f"{device_dict_1[source_mac]['device_name']} "
        f"({source_mac})"
    )

    device_info[source_mac] = {
        "device_name": device_dict_1[source_mac]["device_name"],
        "traffic_summary": agg_df
    }

    display(agg_df.sort_values("dst_type"))

Traffic from Smart Things (d0:52:a8:00:67:5e)


,eth.dst,packet_count,total_bytes,dst_type,packet_count_pct,total_bytes_pct
2,ff:ff:ff:ff:ff:ff,6113,379188,broadcast,2.102616,1.826085
1,14:cc:20:51:33:ea,284620,20385886,unicast,97.897384,98.173915


Traffic from Amazon Echo (44:65:0d:56:cc:d3)


,eth.dst,packet_count,total_bytes,dst_type,packet_count_pct,total_bytes_pct
5,ff:ff:ff:ff:ff:ff,3029,129104,broadcast,0.430932,0.158794
0,01:00:5e:00:00:16,1847,99762,multicast,0.262770,0.122705
2,01:00:5e:7f:ff:fa,1556,241958,multicast,0.221370,0.297602
1,01:00:5e:00:00:fb,37,10505,multicast,0.005264,0.012921
4,14:cc:20:51:33:ea,696426,80821261,unicast,99.079663,99.407978


Traffic from Netatmo Welcome (70:ee:50:18:34:43)


,eth.dst,packet_count,total_bytes,dst_type,packet_count_pct,total_bytes_pct
26,ff:ff:ff:ff:ff:ff,489,30273,broadcast,0.133207,0.017398
8,33:33:ff:18:34:43,13310,1144228,multicast,3.625725,0.657610
22,33:33:ff:df:f4:b7,1829,157286,multicast,0.498231,0.090395
18,33:33:ff:c0:e9:84,1802,154956,multicast,0.490876,0.089056
11,33:33:ff:39:52:13,1578,135700,multicast,0.429857,0.077989
16,33:33:ff:ad:f9:64,1397,120126,multicast,0.380551,0.069039
9,33:33:ff:27:7a:6e,1244,106968,multicast,0.338873,0.061477
14,33:33:ff:6a:dc:11,1128,97000,multicast,0.307274,0.055748
10,33:33:ff:34:9e:1e,1100,94576,multicast,0.299647,0.054355
12,33:33:ff:62:42:ec,1040,89432,multicast,0.283302,0.051398


... (52 outputs left out)

Traffic from MacBook/Iphone (f4:5c:89:93:cc:85)


,eth.dst,packet_count,total_bytes,dst_type,packet_count_pct,total_bytes_pct
8,ff:ff:ff:ff:ff:ff,97,15592,broadcast,1.007374,0.720011
2,01:00:5e:7f:ff:fa,40,8680,multicast,0.415412,0.400827
1,01:00:5e:00:00:fb,34,10083,multicast,0.353100,0.465615
0,01:00:5e:00:00:02,1,46,multicast,0.010385,0.002124
5,33:33:ff:2c:0b:11,1,86,multicast,0.010385,0.003971
6,33:33:ff:93:cc:85,1,86,multicast,0.010385,0.003971
4,14:cc:20:51:33:ea,9447,2130109,unicast,98.109876,98.364690
7,70:5a:0f:e4:9b:c0,8,840,unicast,0.083082,0.038790


Traffic from TPLink Router Bridge LAN (Gateway) (14:cc:20:51:33:ea)


,eth.dst,packet_count,total_bytes,dst_type,packet_count_pct,total_bytes_pct
47,ff:ff:ff:ff:ff:ff,27243,1170016,broadcast,0.221068,1.209959e-02
15,33:33:00:00:00:0c,195685,97955690,multicast,1.587920,1.012998e+00
7,01:00:5e:7f:ff:fa,64883,31252872,multicast,0.526505,3.231982e-01
13,33:33:00:00:00:01,14539,1370866,multicast,0.117979,1.417666e-02
22,33:33:ff:51:33:ea,13465,1157990,multicast,0.109264,1.197523e-02
16,33:33:00:01:00:02,13465,1157990,multicast,0.109264,1.197523e-02
17,33:33:00:01:00:03,13464,1157904,multicast,0.109256,1.197434e-02
19,33:33:ff:00:00:01,13464,1157904,multicast,0.109256,1.197434e-02
18,33:33:ff:00:00:00,13385,1151110,multicast,0.108615,1.190408e-02
14,33:33:00:00:00:02,13373,1150078,multicast,0.108518,1.189341e-02


# Identify router

In [ ]:
# Apply rule to identify router, unicast traffic should be directed to 90% of devices in the network. 
# If a device sends unicast traffic to more than 90% of the devices, it is likely a router.

print("Identifying potential routers based on unicast traffic...")
for mac in device_info:
    # Take off the current mac
    traffic_summary = device_info[mac]["traffic_summary"]
    # Only count devices that are in the device_dict_1 to avoid counting external devices
    traffic_summary = traffic_summary[traffic_summary["eth.dst"].isin(device_dict_1.keys())]
    unicast_count = traffic_summary[traffic_summary["dst_type"] == "unicast"].shape[0]
    total_devices = len(device_dict_1)
    if unicast_count / total_devices > 0.9:
        print(
            f"Device {device_info[mac]['device_name']} ({mac}) "
            f"is likely a router (unicast to {unicast_count}/{total_devices} devices)"
        )

# Or the device with the highest number of unicast destinations is likely the router. (top k)

Identifying potential routers based on unicast traffic...
Device TPLink Router Bridge LAN (Gateway) (14:cc:20:51:33:ea) is likely a router (unicast to 30/31 devices)


- The above rule is quite naive, not gonna work in a setting that have multiple router
- The second rule is:
    - A device communicates with at least 30% of distinct observed devices in the network
        - Why 30%? I am assuming different devices gonna distribute evenly across routers. Usually IoT devices are fixed and will usually connect to the same router as always. However, mobile devices (phone, laptop) can move and connect to different router.
            - In a home settings, we don't usually have multiple routers, so the above rule should work. Otherwise, in a home settings, there are high chances that all devices in the network should connect to all router so that the bottom rule should also work
            - In a corporate settings, I'd say there are barely iot devices, but non-iot devices. Apart from working stations connecting to the same router, there are also plenty of mobile devices connecting to different access point, so that 30% devices coverage might be a reasonable metrics
    - For all devices connected to the candidate device, if the candidate device is an actual router, the connected device is very likely to communicate with this candidate device a lot -> packet count, total bytes should be the most across all devices this device is connecting to
        - In this dataset, there are some edge case. For example: 
            - Samsung Smart Cam (00:16:6c:ab:6b:88) sends the most packet to the router, but send the most data to belkin wemo motion sensor
            - Belkin wemo motion sensor (ec:1a:59:83:28:11) sends the most packet and data to its family devices Belkin wemo switch
        - That's why. I have a threshold saying if 80% of devices connected to the candidate satisfying the above condition, the candidate device is very likely a router. 

In [15]:
print(
    "Identifying potential routers based on unicast traffic "
    "and its corresponding traffic rates..."
)

potential_routers = []

for mac in device_info:

    traffic_summary = device_info[mac]["traffic_summary"]

    # Only consider known devices
    traffic_summary = traffic_summary[
        traffic_summary["eth.dst"].isin(device_dict_1.keys())
    ]

    # Only unicast destinations
    unicast_traffic = traffic_summary[
        traffic_summary["dst_type"] == "unicast"
    ]

    unicast_count = len(unicast_traffic)
    total_devices = len(device_dict_1)

    # Candidate must communicate with >30% of known devices
    if unicast_count / total_devices <= 0.3:
        continue

    is_router = True
    device_satisfying_condition = 0

    for dst_mac in unicast_traffic["eth.dst"]:

        # We need traffic information for the destination device
        if dst_mac not in device_info:
            is_router = False
            break

        dst_traffic_summary = device_info[dst_mac]["traffic_summary"]

        # Only consider unicast traffic for this comparison
        dst_unicast_traffic = dst_traffic_summary[
            dst_traffic_summary["dst_type"] == "unicast"
        ]

        # Find traffic from dst_mac -> current candidate mac
        traffic_to_candidate = dst_unicast_traffic[
            dst_unicast_traffic["eth.dst"] == mac
        ]

        if traffic_to_candidate.empty:
            is_router = False
            break

        packet_pct = traffic_to_candidate["packet_count_pct"].iloc[0]
        bytes_pct = traffic_to_candidate["total_bytes_pct"].iloc[0]

        # Highest percentages among unicast destinations only
        max_packet_pct = dst_unicast_traffic["packet_count_pct"].max()
        max_bytes_pct = dst_unicast_traffic["total_bytes_pct"].max()

        if (
            packet_pct == max_packet_pct
            and bytes_pct == max_bytes_pct
        ):
            device_satisfying_condition += 1
        else:
            print(f"DEBUG: mac: {mac} - dst_mac not satisfying condition - dst_mac: {dst_mac}")

    if device_satisfying_condition < unicast_count * 0.8:
        is_router = False

    if is_router:
        potential_routers.append(mac)

        print(
            f"Potential router: "
            f"{device_info[mac]['device_name']} ({mac}) "
            f"- communicates with "
            f"{unicast_count}/{total_devices} devices "
            f"({unicast_count / total_devices:.2%}), "
            f"device satisfying condition: {device_satisfying_condition}/{unicast_count} "
        )

Identifying potential routers based on unicast traffic and its corresponding traffic rates...
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 14:cc:20:51:33:ea
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 00:16:6c:ab:6b:88
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 00:62:6e:51:27:2e
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: f4:f2:6d:93:51:f1
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 74:2f:68:81:69:42
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 74:c6:3b:29:d7:1d
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: 50:c7:bf:00:56:39
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: d0:73:d5:01:83:08
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condition - dst_mac: ec:1a:59:83:28:11
DEBUG: mac: 08:21:ef:3b:fc:e3 - dst_mac not satisfying condi

# Identify IoT devices

- First approach is device with less than 3 unicast destination is likely an iot device.
- Multifunction purpose devices such as mobile phone, laptop,... usually make SSDP (Simple Service Discovery Protocol), making multicast call to search for local services
    - So, any devices that doesn't make any single multicast calls and only communicate with the router is very likely an IoT device?

In [16]:
# Device with less than 3 unicast destinations is likely a iot device
print("Identifying potential IoT devices based on unicast traffic...")
false_positive = []
false_negative = []
for mac in device_info:
    traffic_summary = device_info[mac]["traffic_summary"]
    # Only count devices that are in the device_dict_1 to avoid counting external devices
    traffic_summary = traffic_summary[traffic_summary["eth.dst"].isin(device_dict_1.keys())]
    unicast_count = traffic_summary[traffic_summary["dst_type"] == "unicast"].shape[0]
    
    if unicast_count <= 3:
        print(
            f"{device_info[mac]['device_name']} ({mac}) is predicted IoT - Unicast to {unicast_count} devices. - Actual is_iot: {device_dict_1[mac]['is_iot_device'] == 1}"
        )
        if device_dict_1[mac]['is_iot_device'] == 0:
            false_positive.append(device_dict_1[mac]["device_name"])
    # else:
    #     print(
    #         f"{device_info[mac]['device_name']} ({mac}) is predicted not IoT - Unicast to {unicast_count} devices. - Actual is_iot: {device_dict_1[mac]['is_iot_device'] == 1}"
    #     )
    #     if device_dict_1[mac]['is_iot_device'] == 1:
    #         false_negative.append(device_dict_1[mac]["device_name"])

# print(f"False Positives (predicted IoT but actually not): {false_positive}")
# print(f"False Negatives (predicted not IoT but actually is): {false_negative}")

Identifying potential IoT devices based on unicast traffic...
Smart Things (d0:52:a8:00:67:5e) is predicted IoT - Unicast to 1 devices. - Actual is_iot: True
Amazon Echo (44:65:0d:56:cc:d3) is predicted IoT - Unicast to 1 devices. - Actual is_iot: True
Netatmo Welcome (70:ee:50:18:34:43) is predicted IoT - Unicast to 2 devices. - Actual is_iot: True
Dropcam (30:8c:fb:2f:e4:b2) is predicted IoT - Unicast to 1 devices. - Actual is_iot: True
Insteon Camera Wireless (e8:ab:fa:19:de:4f) is predicted IoT - Unicast to 3 devices. - Actual is_iot: True
Withings Smart Baby Monitor (00:24:e4:11:18:a8) is predicted IoT - Unicast to 1 devices. - Actual is_iot: True
TP-Link Smart plug (50:c7:bf:00:56:39) is predicted IoT - Unicast to 3 devices. - Actual is_iot: True
iHome (74:c6:3b:29:d7:1d) is predicted IoT - Unicast to 2 devices. - Actual is_iot: True
NEST Protect smoke alarm (18:b4:30:25:be:e4) is predicted IoT - Unicast to 1 devices. - Actual is_iot: True
Netatmo weather station (70:ee:50:03:b8:

# September 10th
- Outgoing + incoming traffic?
- Instead of random threshold 30%, might be looking for a group of device that has much higher traffic compared to the other 
- Might be generate mock MAC router by splitting the traffic of the actual traffic
- Clustering?